### Cronograma

| Etapa | Descrição | Resultado esperado |
|---|---|---|
| 0 | Definição do benchmark | Problema de referência, parâmetros e critérios de validação |
| 1 | Formulação matemática | Equações, hipóteses, domínio e condições de contorno |
| 2 | Discretização espacial | Construção manual dos operadores de diferenças finitas 2D |
| 3 | Construção do sistema linear | Montagem de \(A\eta=b\) e verificação da esparsidade |
| 4 | Análise matricial | Simetria, positividade definida, autovalores e condicionamento |
| 5 | Gradiente Conjugado | Implementação manual do CG |
| 6 | Precondicionamento | Implementação de Jacobi e IC(0) |
| 7 | Experimentos numéricos | Refinamento de malha e heterogeneidade dos coeficientes |
| 8 | Análise dos resultados | Comparações, gráficos, desempenho e conclusões numéricas |

# Etapa 0 - Definição  do Problema e Benchmark

De início, é necessário definir uma referência que permita testar e, idealmente, demonstrar que a implementação está correta. Vale ressaltar que isso é composto por dois objetivos complementares e diferentes: validar a formulação física e formular o algoritmo de solução do sistema linear.

Como referência principal será utilizada ["Semi-implicit finite difference methods for the two-dimensional shallow water equations" (Casulli, 1990)](https://doi.org/10.1016/0021-9991(90)90091-E), que estabelece uma abordagem matemática e algorítmica para simuladores hidrodinâmicos, abordando a resolução numérica das Equações de Águas Rasas Bidimensionais (2D Shallow Water Equations - SWE).

Esse sistema de equações descreve o escoamento de um fluido incompressível com superfície livre (como oceanos costeiros, estuários, grandes lagos, rios largos, mares continentais) sob a hipótese hidrostática (a pressão em qualquer ponto da coluna d'água é determinado exclusivamente pelo peso da coluna de fluido acima dele, ignorando os efeitos das acelerações verticais). Essa hipótese é válida quando a escala horizontal do escoamento (comprimento de onda) é muito maior que a profundidade vertical da coluna d'água,

$$\frac{H}{L} \ll 1$$

onde $H$ é a escala de profundidade vertical e $L$ a escala de comprimento horizontal característico (daí "águas rasas"). Em casos como, por exemplo, perto de pilastras de pontes, situações com forte turbulência vertical, convecção profunda com aceleração vertical do fluido, exigem o uso das equações de Navier-Stokes tridimensionais.

As SWE formam um sistema quasilenear hiperbólico de equações diferenciais parciais em três variáveis independentes $(x,y,t)$ ("quasilinear" porque é linear em relção às derivadas de ordem mais alta, mas os coeficientes dessas derivadas podem  depender das variáveis independentes e das próprias variáveis dependentes desconhecidas; "hiperbólico" porque o problema de valor inicial é bem posto, no sentido de o problema possuir solução única que depende continuamente dos dados iniciais e todas as perturbações iniciais se propagam a velocidades reais e finitas ao longo das curvas e superfícies características):

$$
\begin{array}{lc}
  \dfrac{\partial u}{\partial t} + u\dfrac{\partial u}{\partial x} + v\dfrac{\partial u}{\partial y} = -g\dfrac{\partial z}{\partial x} -\gamma u + \tau_x & \qquad (1) \\[8pt]
  \dfrac{\partial v}{\partial t} + u\dfrac{\partial v}{\partial x} + v\dfrac{\partial v}{\partial y} = -g\dfrac{\partial z}{\partial y} -\gamma v + \tau_y & \qquad (2) \\[8pt]
  \dfrac{\partial z}{\partial t} + \dfrac{\partial [(h+z)u]}{\partial x} + \dfrac{\partial [(h+z)v]}{\partial y} = 0 & \qquad (3)
\end{array}
$$

onde:

* $u(x,y,t)$ e $v(x,y,t)$ são as componentes da velocidade do fluido mediadas na profundidade, respectivamente nas direções $x$ e $y$;
* $z(x,y,t)$ é a elevação da superfície livre da água medida a partir de um nível de referência não perturbado;
* $h(x,y)$ é a profundidade da água também medida a partir de um nível de referência não perturbado;
* $g$, constante de aceleração da gravidade;
* $\tau_x$ e $\tau_y$ são os termos de estresse do vento nas direções $x$ e $y$, respectivamente;
* $\gamma$ é o coeficiente de atrito com o fundo, dado pela equação,

$$\gamma = \frac{g\sqrt{u^2 + v^2}}{C_z^2H}$$

sendo, $H(x,y,t) = h(x,y) + z(x,y,t)$ a profundidade total da água e $C_z$ o coeficiente de fricção de Chezy, que mede a capacidade de resistência ao escoamento imposta pelo leito sobre a velocidade média na vertical $(u,v)$ e a profundidade total $H$.

<figure align="center">
  <figcaption><strong>Figura 1: Domínio Físico, Escalas e Variáveis das Equações de Águas Rasas Bidimensionais (2D SWE)</strong></figcaption>
  <img src="../figures/swe_physical_domain.svg?v=4" alt="Domínio Físico, Escalas e Variáveis das Equações de Águas Rasas Bidimensionais (2D SWE)" width="100%"/>
  <figcaption><small>Fonte: Produzido pelo Autor.</small></figcaption>
</figure>

Definindo o vetor velocidade horizontal $\vec{u} = (u, v)$ e o vetor de forçamento externo $\vec{\tau} = (\tau_x, \tau_y)$, usando o operador diferencial vetorial $\nabla = \left( \frac{\partial}{\partial x}, \frac{\partial}{\partial y}\right)$, temos as equações de águas rasas na forma semi-compacta

$$
\begin{array}{lc}
  \dfrac{\partial u}{\partial t} + \vec{u}\cdot \nabla u = -g(\nabla z)_x - \gamma u + \tau_x  & \qquad (4) \\[8pt]
  \dfrac{\partial v}{\partial t} + \vec{u}\cdot \nabla v = -g(\nabla z)_y - \gamma v + \tau_y & \qquad (5) \\[8pt]
  \dfrac{\partial z}{\partial t} + \nabla \cdot [(h + z)\vec{u}] = 0 & \qquad (6)
\end{array}
$$

e, em forma compacta total,

$$
\begin{array}{lc}
  \dfrac{\partial \vec{u}}{\partial t} + (\vec{u}\cdot \nabla)\vec{u} = -g\nabla z - \gamma \vec{u} + \vec{\tau} & \qquad (7) \\[8pt]
  \dfrac{\partial z}{\partial t} + \nabla \cdot [(h + z)\vec{u}] = 0 & \qquad (8) \\[8pt]
\end{array}
$$

### 0.1. Significado das equações

Na primeira equação $(1)$,

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y} = -g\frac{\partial z}{\partial x} -\gamma u + \tau_x$$

no lado esquerdo, temos um conjunto de derivadas parciais que medem a mudança da velocidade $u$ no tempo $\frac{\partial u}{\partial t}$ e no campo espacial $x \times y$ onde $u$ varia: $\frac{\partial u}{\partial x}$ e $\frac{\partial u}{\partial y}$. No lado direito, sobre o termo $-g\frac{\partial z}{\partial x}$, como, pelo princípio hidrostático a pressão varia unicamente em função da coluna de água, a pressão horizontal depende da inclinação da superfície livre $z(x,y,t)$:

* Se a superfície da água está mais alta à esquerda e mais baixa a direita, então $\frac{\partial z}{\partial x} < 0$ e $-g\frac{\partial z}{\partial x} > 0$: a gravidade desloca a água da posição mais alta para a mais baixa, acelerando $u$ para a direita.
* Este termo impulsiona ou freia conforme o sinal da inclinação $\frac{\partial z}{\partial x}$.

Ainda na direita, o termo $-\gamma u$ atua como dissipação/amortecimento por atrito, opondo-se ao movimento, dado $\gamma \ge 0$. Por fim, $\tau_x$ um forçamento externo pelo vento na superfície, podendo acelerar ou desacelerar conforme a direção do vento.

<figure align="center">
  <figcaption><strong>Figura 2: Significado Físico e Dinâmico da Equação de Quantidade de Movimento na Direção x</strong></figcaption>
  <img src="../figures/swe_momentum_x_meaning.svg?v=4" alt="Significado Físico e Dinâmico da Equação de Quantidade de Movimento na Direção x" width="100%"/>
  <figcaption><small>Fonte: Produzido pelo Autor.</small></figcaption>
</figure>

Na segunda equação $(2)$,

$$\frac{\partial v}{\partial t} + u\frac{\partial v}{\partial x} + v\frac{\partial v}{\partial y} = -g\frac{\partial z}{\partial y} -\gamma v + \tau_y$$

a lógica é exatamente a mesma, mas para a componente $v$ na horizontal $y$.

A terceira equação $(3)$,

$$\frac{\partial z}{\partial t} + \frac{\partial [(h+z)u]}{\partial x} + \frac{\partial [(h+z)v]}{\partial y} = 0$$

descreve o sobe-e-desce $\frac{\partial z}{\partial t}$ com conservação de massa para um fluido incomprensível, ou seja, sem evaporação, sem chuva ou infiltração, daí ser igual a $0$. Podemos reescrevê-la como

$$\frac{\partial x}{\partial t} = -\left(\frac{\partial [(h + z)u]}{\partial x} + \frac{\partial [(h + z)v]}{\partial y}\right)$$

onde $(h + z)x$ e $(h + z)v$ são os fluxos volumétricos por unidade de largura nas direções $x$ e $y$, ou seja, a profundidade total vezes a velocidade. Se entra mais água do que sai na coluna d'água, o divergente do fluxo é negativo, o lado direito torna-se positivo e $\frac{\partial z}{\partial t} > 0$, logo, a coluna de água sobe; se o gradiante for positivo, a coluna de água desce.

<figure align="center">
  <figcaption><strong>Figura 3: Significado Físico da Equação da Continuidade (Conservação de Massa e Sobe-e-Desce da Superfície Livre)</strong></figcaption>
  <img src="../figures/swe_continuity_meaning.svg?v=2" alt="Significado Físico da Equação da Continuidade (Conservação de Massa e Sobe-e-Desce da Superfície Livre)" width="100%"/>
  <figcaption><small>Fonte: Produzido pelo Autor.</small></figcaption>
</figure>

### 0.2. Análise de Características

Quando uma superfície fluida em equilíbrio sofre uma perturbação inicial, seja no fluxo, seja na elevação, dois fenômenos ocorrem simultaneamente:

1. O próprio fluido se desloca (advecção) com velocidade $\vec{u} = (u, v)$.
2. Uma onda de gravidade superficial se espalha circularmente a partir do ponto de perturbação com velocidade $c = \sqrt{gH}$.

Em determinados cenários, como oceanos e canais, por exemplo, a velocidade da onda $c = \sqrt{gH}$ pode ser várias ordens de magnitude maior do que a velocidade da corrente ($|u|$, $|v|$). Para lidar com a diferença de escalas em uma simulação é necessário responder duas perguntas:

* Quais são exatamente todas as velocidades e direções de propagação de perturbação/informação contidas nas equações?
* Quais termos específicos da equação produzem as ondas rápidas ($\sqrt{gH}$) e quais produzem o transporte lento ($\vec{u}$)?

Separar matematicamente esses dois comportamentos torna a simulação mais eficiente. Em vez de usar um método computacional único, talvez pesado, para resolver a equação inteira, o ideal é separar um método apenas para os termos das ondas rápidas, responsáveis pela restrição de estabilidade numérica em esquemas explícitos (conforme analisado no contexto de diferenças finitas e do Teorema da Equivalência de Lax em Atkinson e Han, [Capítulo 6](https://link.springer.com/chapter/10.1007/978-1-4419-0458-4_6 "Atkinson Han Theoretical Numerical Analysis Chapter 6 Finite Difference Method")), e outro método mais simples para o transporte lento.

Para identificar essas velocidades de propagação de forma sistemática, reescrevemos o sistema como um sistema hiperbólico de primeira ordem na forma matricial (Evans, [Partial Differential Equations, Capítulo 7, Seção 7.3](https://www.cs.mcgill.ca/~akroit/math/diffeq/Evans%20Partial%20Differential%20Equations.pdf "Evans Partial Differential Equations AMS Bookstore")),

$$\frac{\partial W}{\partial t} + A(W)\frac{\partial W}{\partial x} + B(W)\frac{\partial W}{\partial y} = D(W)$$

onde, para visualizar como as equações $(1), (2) \text{ e } (3)$ se organizam nessa notação, define-se o vetor $W = (u, v, z)^T$ e a profundidade total $H = h + z$ e, empilhando os termos das equações de momento e de continuidade, o sistema toma a forma explícita:

$$
\frac{\partial}{\partial t} \begin{pmatrix} u \\ v \\ z \end{pmatrix} + \underbrace{\begin{pmatrix} u & 0 & g \\ 0 & u & 0 \\ H & 0 & u \end{pmatrix}}_{A(W)} \frac{\partial}{\partial x} \begin{pmatrix} u \\ v \\ z \end{pmatrix} + \underbrace{\begin{pmatrix} v & 0 & 0 \\ 0 & v & g \\ 0 & H & v \end{pmatrix}}_{B(W)} \frac{\partial}{\partial y} \begin{pmatrix} u \\ v \\ z \end{pmatrix} = \underbrace{\begin{pmatrix} -\gamma u + \tau_x \\ -\gamma v + \tau_y \\ -u \frac{\partial h}{\partial x} - v \frac{\partial h}{\partial y} \end{pmatrix}}_{D(W)}
$$

* $\frac{\partial W}{\partial t}$ é o vetor das derivadas temporais $\left[\frac{\partial u}{\partial t}, \frac{\partial v}{\partial t}, \frac{\partial z}{\partial t}\right]^T$.
* $\frac{\partial W}{\partial x}$ e $\frac{\partial W}{\partial y}$ são os vetores das derivadas espaciais em $x$ e $y$.
* $A(W)$ e $B(W)$ são matrizes $3 \times 3$ que multiplicam $\frac{\partial W}{\partial x}$ e $\frac{\partial W}{\partial y}$, respectivamente. Elas dependem de $W$ e carregam a dinâmica de propagação (advecção e ondas). Estas duas matrizes mostram como a variação de uma grandeza influencia as outras, onde $u$ e $v$ na diagonal principal representam o transporte (advecção), enquanto $g$ e $H$ fora da diagonal acoplam altura da água com a velocidade, daí modelando as ondas.
* $D(W)$ é o vetor com os termos que não multiplicam derivadas de $W$, como atrito de fundo ($-\gamma u$, $-\gamma v$), vento ($\tau_x$, $\tau_y$) e a inclinação de fundo fixo $\left(\frac{\partial h}{\partial x} \text{ e } \frac{\partial h}{\partial y}\right)$. 

Com o sistema nesse formato, analisa-se como uma perturbação se espalha pelo espaço $(x, y)$ ao longo do tempo $t$ (espaço 3D $x \times y \times t$), a partir de um ponto qualquer, digamos $(0, 0)$ no instante $t = 0$, sendo que a onda se espalha em círculos cada vez maiores conforme o tempo sobe. Vale destacar que nesse sistema uma perturbação localizada não afeta o espaço inteiro instantaneamente. Empilhando-se esses círculos ao longo do tempo $t$, forma-se geometricamente um cone no espaço-tempo. A superfície do cone característico demarca a fronteira até onde a informação da perturbação conseguiu chegar (Evans, [Partial Differential Equations, Capítulo 3, Seção 3.2](https://www.cs.mcgill.ca/~akroit/math/diffeq/Evans%20Partial%20Differential%20Equations.pdf "Evans Partial Differential Equations AMS Bookstore")). A partir disso, chamando de $\phi(t, x, y) = \text{constante}$ uma função para a superfície desse cone, o gradiente espaço-temporal de $\phi$:

$$\mathbf{n} =\left(\frac{\partial \phi}{\partial t}, \frac{\partial \phi}{\partial x}, \frac{\partial \phi}{\partial y}\right) = (q, r, s)$$

é um vetor normal (perpendicular) à superfície desse cone espaço-temporal, com $(r, s)$ apontando na direção espacial $(x, y)$ e $q$ apontando na direção temporal $t$.

<figure align="center">
  <figcaption><strong>Figura 4: Cone Característico no Espaço-Tempo (x, y, t) e Propagação Circular da Onda no Plano Espacial (x, y)</strong></figcaption>
  <img src="../figures/swe_characteristic_cone.svg?v=7" alt="Cone Característico no Espaço-Tempo (x, y, t) e Propagação Circular da Onda no Plano Espacial (x, y)" width="100%"/>
  <figcaption><small>Fonte: Produzido pelo Autor.</small></figcaption>
</figure>

Pela teoria de sistemas hiperbólicos (Evans, [Capítulo 7, Seção 7.3](https://www.cs.mcgill.ca/~akroit/math/diffeq/Evans%20Partial%20Differential%20Equations.pdf "Evans Partial Differential Equations AMS Bookstore")) e pela formulação do polinômio característico de combinações matriciais (Horn e Johnson, [Matrix Analysis, Capítulo 1](https://www.cambridge.org/core/books/matrix-analysis/9CF2CB491C9E97948B15FAD835EF9A8B "Horn Johnson Matrix Analysis Chapter 1 Eigenvalues")), uma onda ou frente de informação só consegue viajar ao longo de uma superfície no espaço-tempo se a condição

$$\det(qI + rA + sB) = 0$$

for satisfeita. Resolvendo esse determinante, obtemos a equação característica resultante e, fatorando-se, em dois componentes distintos:

1. A equação $q + ru + sv = 0$: representa o transporte puramente advectivo (sem propagação de onda relativa ao fluido), definindo a trajetória de uma partícula ao longo do eixo do cone com velocidade $\vec{u} = (u, v)$.
2. A equação $(q + ru + sv)^2 - gH(r^2 + s^2) = 0$, que define a superfície do cone e a propagação radial de ondas superficiais com velocidade $c = \sqrt{gH}$.

A abordagem proposta por [Casulli](https://doi.org/10.1016/0021-9991(90)90091-E) consistiu em identificar quais termos nas matrizes $A$ e $B$ geram o termo $gH$ na equação característica, que são:

* O gradiente de elevação da superfície livre $(\nabla z)$ nas equações de momento.
* O divergente de velocidade $(\nabla \cdot \vec{u})$ na equação da continuidade.

### 0.3. Método Numérico Proposto

Casulli projeta um esquema de Diferenças Finitas Semi-Implícito sobre uma malha espacial escalonada, onde $z_{i, j}$ é avaliado no centro da célula, $u_{i+1/2, j}$ nas faces verticais e $v_{i, j+1/2}$ nas faces horizontais, evitando oscilações espúrias de pressão/elevação. Podemos imaginar dividir a superfície da água em células de uma grade. Se guardarmos todas as variáveis $(u, v, z)$ no mesmo ponto, centro do quadrado, para calcular a inclinação da água $\frac{\partial z}{\partial x}$ no centro $i$, usamos os vizinhos $i + 1$ e $i - 1$: $\frac{z_{i+1} - z_{i-1}}{2\Delta x}$. Porém, se a superfície da água ficar alternando rapidamente entre alto e baixo, a diferença entre $i + 1$ e $i - 1$ dá zero, sendo que a equação não consegue modelar esse serrilhado falso (oscilação espúria). A solução consiste em a elevação $z_{i,j}$ fica no centro da célula; a velocidade horizontal $u_{i+1/2, j}$ fica nas faces verticais esquerda e direita, exatamente entre dois centros $z_{i,j}$ e $z_{i+1, j}$; a velocidade vertical $v_{i, j+1/2}$ fica nas faces horizontais de cima e de baixo; assim, o gradiente $\frac{z_{i+1, j} - z_{i, j}}{\Delta x}$ usa células imediatamente adjacentes, eliminando qualquer ponto cego.

<figure align="center">
  <figcaption><strong>Figura 5: Malha Espacial Escalonada de Casulli vs. Malha Colocalizada (Eliminação de Oscilações Espúrias e do Ponto Cego)</strong></figcaption>
  <img src="../figures/swe_staggered_grid.svg?v=1" alt="Malha Espacial Escalonada de Casulli vs. Malha Colocalizada (Eliminação de Oscilações Espúrias e do Ponto Cego)" width="100%"/>
  <figcaption><small>Fonte: Produzido pelo Autor.</small></figcaption>
</figure>

O método opera em três pilares estruturais:

A: Tratamento Semi-Implícito Linearizado

Para avançar do tempo atual $k$ conhecido para o próximo passo $k + 1$ desconhecido:

* Termos Implícitos no tempo $k + 1$: o gradiente de elevação $\nabla z^{k+1}$, o divergente da velocidade $\nabla \cdot (\vec{u})^{k+1}$ e o termo linear do atrito com o fundo $\gamma^k (\vec{u})^{k+1}$, termos que geram as ondas rápidas. Isso remove a restrição severa de passo de tempo $\Delta t$ das ondas gravitacionais.
* Termos Explícitos no tempo $k$: usamos os valores que já conhecemos. A advecção $F$, a profundidade $H$ e o coeficiente de atrito $\gamma^k$ ficam congelados no tempo $k$.
* Linearidade: como o atrito real é quadrático, dado que $\gamma$ depende de $\sqrt{u^2 + v^2}$. Se deixássemos $\gamma$ no temp $k + 1$ o sistema ficaria não linear. Ao escrever $\gamma^ku^{k+1}$, congelamos o coeficiente $\gamma$ no passo anterior e deixamos apenas $u^{k+1}$ como incógnita e mantendo o sistema linear.

Ao substituir algebricamente as equações discretas de momento $u^{k+1}$ e $v^{k+1}$ diretamente dentro da equação discreta de continuidade, as velocidades são eliminadas e obtém-se um único sistema linear pentadiagonal onde apenas as elevações $z_{i,j}^{k+1}$ são incógnitas.

* Desde que a coluna d'água seja positiva $(H > 0)$, a matriz desse sistema é simétrica, estritamente diagonal dominante e definida positiva (diagonal principal positiva e elementos fora da diagonal negativos), garantindo a existência e unicidade da solução a cada passo de tempo e permitindo o uso de métodos iterativos ultrarrápidos. Uma vez encontrado $z^{k+1}$, as velocidades $(\vec{u})^{k+1}$ são atualizadas explicitamente por substituição direta.

# Etapa 1 - Formulação Matemática

# Etapa 2 - Discretização Espacial

# Etapa 3 - Construção do Sistema Linear

# Etapa 4 - Análise Matricial

# Etapa 5 - Gradiente Conjugado

# Etapa 6 - Precondicionamento

# Etapa 7 - Experimentos Numéricos

# Etapa 8 - Análise dos Resultados